In [ ]:
import csv
import pandas as pd
import numpy as np
import copy
import matplotlib.pyplot as plt
import statsmodels.api as sm
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
import glob
import seaborn as sns
import scipy.stats as stats
#from pymer4.models import Lmer
from statsmodels.stats.diagnostic import het_white
import arviz as az
import bambi as bmb
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.graphics.regressionplots import abline_plot
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

random_seed = 1234
fs = 30

In [ ]:
mainDir = '/Baseline/'

file1 = mainDir + 'ForGLM_NewVsRev.csv'
file2 = mainDir + 'ForGLM.csv'

In [ ]:
dfGLM = pd.read_csv(file2)

scaler = StandardScaler()

scaleColList = ['EatRate', 'DrinkRate', 'PredRate',  'Recency', 'Dwelltime', 'CowCount','Uncertainty'] # ,'PredKillRate']

dfGLM[scaleColList] = scaler.fit_transform(dfGLM[scaleColList])

model = smf.glm(formula = "PatchRevisit ~ EatRate + DrinkRate + PredRate  + Recency + Dwelltime + CowCount + Uncertainty + C(AgentID)", #+ PredKillRate
  
                data = dfGLM, 
                family = sm.families.Binomial())

# Fit the model
result = model.fit()
# Display and interpret results
print(result.summary())
# Estimated default probabilities
predictions = result.predict()

# the independent variables set
X0 = dfGLM[scaleColList]
X = X0.dropna(how='any') 
# VIF dataframe
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns

# calculating VIF for each feature
vif_data["VIF"] = [variance_inflation_factor(X.values, i)
                          for i in range(len(X.columns))]

print(vif_data)

#Normality
fig = plt.figure(figsize = (16, 9))

ax = sns.distplot(result.resid_response, hist = False, kde_kws = {"shade" : True, "lw": 1}, fit = stats.norm)

ax.set_title("KDE Plot of Model Residuals (Blue) and Normal Distribution (Black)")
ax.set_xlabel("Residuals")

## Q-Q PLot

fig = plt.figure(figsize = (11, 9))
ax = fig.add_subplot(111)

sm.qqplot(result.resid_response, dist = stats.norm, line = 's', ax = ax)

ax.set_title("Q-Q Plot")

labels = ["Statistic", "p-value"]

norm_res = stats.shapiro(result.resid_response)

for key, val in dict(zip(labels, norm_res)).items():
    print(key, val)
    
fig = plt.figure(figsize = (11, 9))

het_white_res = het_white(result.resid_response, result.model.exog)

labels = ["LM Statistic", "LM-Test p-value", "F-Statistic", "F-Test p-value"]

for key, val in dict(zip(labels, het_white_res)).items():
    print(key, val)
    
# fit prediction model
#pred = smf.ols('predicted ~ actual', data=new_df).fit()
lenVar = len(scaleColList) 
CI = result.conf_int(alpha=0.05)[1]
tempCI = CI[(len(CI)-lenVar):(len(CI))]
tempCf = result.params[(len(result.params)-lenVar):(len(result.params))]

error = tempCI - tempCf

coefList = []
errorList = []
for iii in range(lenVar):
    coefList.append(tempCf[iii])
    errorList.append(error.iloc[iii])
fig = plt.figure(figsize = (28, 9))
fs = 30
plt.bar(scaleColList, coefList)
plt.errorbar(scaleColList,coefList,yerr=errorList,fmt='o', ecolor='red', capsize=15,elinewidth=10,capthick=10)
plt.title("Factors that influence patch revisitation choice", fontsize = fs+4, fontweight="bold")
plt.axhline(y=0, color=".5")
plt.xlabel("Variable Category for Chosen Patch Relative to Non-chosen", fontsize = fs-2, labelpad=15, fontweight="bold")
plt.ylabel("Standardized GLM coefficient values", fontsize = fs, fontweight="bold")
#plot.errorbar(coefs,yerr=error, fmt="o", color="r")
plt.xticks(fontsize=fs-4)
plt.yticks(fontsize=fs-8)
plt.subplots_adjust(left=0.3)
plt.show()
